In [1]:
cd /root
cd hardware_rounding_error_predictor

# --- env and PDFs ---
apt-get install -y poppler-utils
mkdir -p /workspace
ln -sfn /root/hardware_rounding_error_predictor/literature /workspace/literature
export HF_HOME=/root/hf_cache MUFU_CACHE_DIR=/root/mufu_cache
mkdir -p /root/hf_cache /root/mufu_cache

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libnspr4 libnss3 libpoppler134 poppler-data
Suggested packages:
  ghostscript fonts-japanese-mincho | fonts-ipafont-mincho
  fonts-japanese-gothic | fonts-ipafont-gothic fonts-arphic-ukai
  fonts-arphic-uming fonts-nanum
The following NEW packages will be installed:
  libnspr4 libnss3 libpoppler134 poppler-data poppler-utils
0 upgraded, 5 newly installed, 0 to remove and 60 not upgraded.
Need to get 4952 kB of archives.
After this operation, 22.2 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/main amd64 poppler-data all 0.4.12-1 [2060 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble/main amd64 libnspr4 amd64 2:4.35-1.1build1 [117 kB]
Get:3 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 libnss3 amd64 2:3.98-1ubuntu0.1 [1445 kB]
Get:4 http://archive.ubuntu.com/ubuntu noble-updates/main am

In [2]:
# --- catalog + demo ---
python3 build_catalog.py --seq-lens 100 250 1000 4000 --no-verify 2>&1 | tee /root/catalog_build.log

cat > /root/demo.sh << 'EOF'
#!/bin/bash
cd /root/hardware_rounding_error_predictor
export HF_HOME=/root/hf_cache MUFU_CACHE_DIR=/root/mufu_cache
for SEQ in 100 250 1000 4000; do
    echo "================ seq=$SEQ ================"
    python3 ffn_chain_test.py all $SEQ
    echo "---- cuBLAS mode ----"
    EMULATOR_TARGET=cublas python3 ffn_chain_test.py compare $SEQ
done
EOF

M grid restricted to 4 specified seq lengths: [100, 250, 1000, 4000]

=== down_proj (N=2560, K=9728) ===
  [ 1/ 4] M=  100 N= 2560 K= 9728 /venv/main/lib/python3.12/site-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
  recipe=split_k_workspace_outtype    split_k=3  verify=skipped               kernel=none
  [ 2/ 4] M=  250 N= 2560 K= 9728   recipe=single_walk                  split_k=1  verify=skipped               kernel=none
  [ 3/ 4] M= 1000 N= 2560 K= 9728   recipe=single_walk                  split_k=1  verify=skipped               kernel=none
  [ 4/ 4] M= 4000 N= 2560 K= 9728   recipe=single_walk                  split_k=1  verify=skipped               kernel=none
  2 region(s) in this lane

=== gate_up_proj (N=9728, K=2560) ===
  [ 1/ 4] M=  100 N= 9728 K= 2560   recipe=single_walk                  sp

In [3]:
SKU=$(nvidia-smi --query-gpu=name --format=csv,noheader | head -1 | tr -d ' ')
bash /root/demo.sh 2>&1 | tee /root/ffn_demo_${SKU}.log

================ seq=100 ================
Sequence length override: 100
Data dir: ffn_chain_data/seq100
PHASE 1: INSTRUMENTED MODEL FORWARD PASS

Loading Qwen/Qwen3-4B...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 398/398 [00:01<00:00, 294.32it/s]
Loaded. 36 layers, targeting layer 20.

ARCHITECTURE INSPECTION
------------------------------------------------------------
  model.config.hidden_size:       2560
  model.config.intermediate_size:  9728
  model.config.hidden_act:         silu
  model.config.rms_norm_eps:       1e-06
  LayerNorm type:  Qwen3RMSNorm
  MLP type:        Qwen3MLP
  act_fn type:     SiLUActivation
  gate_proj:       torch.Size([9728, 2560])  bias=False
  up_proj:         torch.Size([9728, 2560])  bias=False
  down_proj:       torch.Size([2560, 9728])  bias=False
  ln weight shape: torch.Size([2560])

TOKENIZATION
------------------------------------------------------------
  Using PDF (pdftotext): /workspace/